# Train PPO on CarRacing-v3 (Kaggle GPU)

This notebook trains a PPO agent with a CNN policy on CarRacing-v3 using Stable-Baselines3.

**Setup:** Settings → Accelerator → GPU T4 x1, Internet → ON

**Output:** `ppo_carracing.zip` (~25 MB) — download it from the Output panel when done,
then place it at `dl/ppo_carracing.zip` in the local repo.

In [ ]:
!pip install -q "gymnasium[box2d]==1.2.3" "stable-baselines3==2.8.0" swig

In [ ]:
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecTransposeImage

TOTAL_STEPS = 500_000   # raise to 1_000_000 for a stronger agent
N_ENVS = 8

vec_env = make_vec_env(
    "CarRacing-v3",
    n_envs=N_ENVS,
    env_kwargs={"continuous": True},
)
vec_env = VecTransposeImage(vec_env)

model = PPO(
    "CnnPolicy",
    vec_env,
    verbose=1,
    learning_rate=1e-4,
    n_steps=512,
    batch_size=128,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.0,
    vf_coef=0.5,
    max_grad_norm=0.5,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

print(f"Training for {TOTAL_STEPS:,} steps on {N_ENVS} parallel envs")
model.learn(total_timesteps=TOTAL_STEPS, progress_bar=False)

model.save("/kaggle/working/ppo_carracing")
print("Saved to /kaggle/working/ppo_carracing.zip")

## Sanity check: run seed 0 with the trained agent

In [ ]:
import gymnasium as gym

env = gym.make("CarRacing-v3", continuous=True)
obs, _ = env.reset(seed=0)
total = 0.0
done = False
while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, term, trunc, _ = env.step(action)
    total += reward
    done = term or trunc
env.close()
print(f"seed=0 return={total:.1f}")

## Quick eval on seeds 0-9

In [ ]:
import numpy as np

model_eval = PPO.load("/kaggle/working/ppo_carracing", device="cuda" if torch.cuda.is_available() else "cpu")

returns = []
for seed in range(10):
    env = gym.make("CarRacing-v3", continuous=True)
    obs, _ = env.reset(seed=seed)
    total = 0.0
    done = False
    while not done:
        action, _ = model_eval.predict(obs, deterministic=True)
        obs, reward, term, trunc, _ = env.step(action)
        total += reward
        done = term or trunc
    env.close()
    returns.append(total)
    print(f"seed={seed}  return={total:.1f}")

print(f"\nmean={np.mean(returns):.1f}  std={np.std(returns):.1f}  "
      f"good(>=700)={sum(r >= 700 for r in returns)}  "
      f"full_lap(>=900)={sum(r >= 900 for r in returns)}")

## Download

The trained checkpoint is at `/kaggle/working/ppo_carracing.zip` in the **Output** panel on the right.

Download it and place it at `dl/ppo_carracing.zip` in the local repo.